In [1]:
!pip install -q trino

In [2]:
from trino.dbapi import connect
from trino.auth import OAuth2Authentication
from redirect_handler import REDIRECT_HANDLER
import urllib3
import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

TRINO_URI = "https://trino-proxy:443"

# Use the Catalog (User 2: Anna)
So far we have just used one user to query our data. We wouldn't need OPA for this.
Lets setup a second user: Anna.

1. Open a new private browser window, open `http://localhost:8181` (Lakekeeper UI) and login using Username `anna` and Password `iceberg`. Anna has no permissions yet. That's alright.
2. Execute the following cell - copy the shown login link into the private browser you used before and re-login as `anna` if asked. The cell execution should fail, as Anna has no permissions yet.
3. In your regular browser, navigate to the Warehouse "demo" and grant anna the "select" permission. Now re-run the cell below. Copy the login link to your private browser tab again.

In [3]:
conn = connect(
    host=TRINO_URI,
    auth=OAuth2Authentication(REDIRECT_HANDLER),
    http_scheme="https",
    verify=False,
    catalog="lakekeeper" # This line is new
)

cur = conn.cursor()
# Execute query and fetch all rows
cur.execute("SELECT * FROM finance.products")
rows = cur.fetchall()

# Get column names
columns = [desc[0] for desc in cur.description]

# Create DataFrame
df = pd.DataFrame(rows, columns=columns)

# Display DataFrame
print(df)

Open the following URL in browser for the external authentication:
http://localhost:38191/oauth2/token/initiate/e7233b4d42a45ac49ed25f17515709395ff9684faaab08cbde5b4897fe97924d


TrinoUserError: TrinoUserError(type=USER_ERROR, name=PERMISSION_DENIED, message="Access Denied: Cannot access catalog lakekeeper", query_id=20251001_211433_00011_8phzu)

As you can see, Trino enforces lakekeeper permissions via OPA!